# Research Pipeline Notebook
## Author: Gemini

This notebook implements a 5-stage research pipeline for analyzing conflict data.

In [1]:
appName = "researchPipeline"
import pandas as pd
import numpy as np
import os

## Stage 1: Ingestion

In this stage, we load the raw ACLED data into the notebook.

In [2]:
data_path = "data/ACLED Data Feb 14 2026.csv"
df = pd.read_csv(data_path)
print(f"Loaded {len(df)} rows.")
df.head()

Loaded 91441 rows.


,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,...,location,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp
0,MMR16462,2021-05-05,2021,1,Strategic developments,Strategic developments,Looting/property destruction,Military Forces of Myanmar (2021-),NaN,State forces,...,Tha Min Chan,22.3606,94.8957,1,Democratic Voice of Burma,National,"Looting: On 5 May 2021, in Tha Min Chan villag...",0,NaN,1620765673
1,MMR16496,2021-05-05,2021,1,Strategic developments,Strategic developments,Looting/property destruction,RCSS/SSA-S: Restoration Council of Shan State/...,PSLF/TNLA: Palaung State Liberation Front/Ta'a...,Rebel group,...,Yae Oe,22.8118,97.3299,1,Irrawaddy,National,"Property destruction: On 5 May 2021, in Yae Oe...",0,NaN,1753971383
2,MMR16128,2021-05-05,2021,1,Political violence,Battles,Armed clash,Military Forces of Myanmar (2021-),NaN,State forces,...,Hpapun,18.0650,97.4449,2,Karen Information Center News,Subnational,"On 5 May 2021, between Hpapun and Khapu, Hpapu...",7,NaN,1753971383
3,PHL12117,2021-05-05,2021,1,Demonstrations,Protests,Peaceful protest,Protesters (Philippines),LIPI: Liga Independencia Pilipinas; LPP: Leagu...,Protesters,...,Makati,14.5502,121.0326,1,Philippine News Agency,National,"On 5 May 2021, anti-communist groups led by th...",0,crowd size=no report,1628035296
4,IDN5951,2021-05-05,2021,1,Demonstrations,Protests,Peaceful protest,Protesters (Indonesia),NaN,Protesters,...,Purworejo,-7.7141,110.0082,1,Detik,National,"On 5 May 2021, a group of people held a peacef...",0,crowd size=no report,1759789823


### Reflection - Stage 1
* **What I learned:** I learned how to efficiently load large CSV datasets using pandas and the importance of verifying the file path relative to the notebook location.
* **Challenge faced:** The dataset contains many columns, making it difficult to visualize the entire structure at once.
* **How I solved it:** I used `df.head()` and `df.columns` to inspect the most relevant fields first.

## Stage 2: Validation & Cleaning

This stage involves detecting and fixing incorrect or missing values to ensure data quality.

In [3]:
# Convert event_date to datetime
df['event_date'] = pd.to_datetime(df['event_date'])

# Handle missing fatalities by filling with 0
df['fatalities'] = df['fatalities'].fillna(0)

# Drop duplicates
df = df.drop_duplicates()

print("Data cleaned and validated.")

Data cleaned and validated.


### Reflection - Stage 2
* **What I learned:** I learned that real-world data often has inconsistent date formats and missing numerical values that can break analysis.
* **Challenge faced:** Determining whether a missing 'fatalities' value meant zero or 'unknown'.
* **How I solved it:** I decided to fill with 0 to maintain numerical consistency while noting that it represents a conservative estimate.

## Stage 3: Transformation

Creating new features to enhance the dataset for deeper analysis.

In [4]:
# Extract temporal features
df['month'] = df['event_date'].dt.month
df['day_of_week'] = df['event_date'].dt.day_name()

# Create high_fatality indicator
df['high_fatality'] = df['fatalities'] > 10

print("Transformation complete. New features added.")

Transformation complete. New features added.


### Reflection - Stage 3
* **What I learned:** I learned how to use the `.dt` accessor in pandas to extract granular time information from datetime objects.
* **Challenge faced:** Creating meaningful categorical groups from continuous fatality numbers.
* **How I solved it:** I implemented a binary 'high_fatality' threshold to quickly identify significant events.

## Stage 4: Aggregation

Summarizing data to compute insights.

In [5]:
# Group by event type and compute stats
summary_stats = df.groupby('event_type').agg({
    'event_id_cnty': 'count',
    'fatalities': ['sum', 'mean']
}).reset_index()

summary_stats.columns = ['event_type', 'event_count', 'total_fatalities', 'average_fatalities']
print("Aggregation complete.")
summary_stats

Aggregation complete.


,event_type,event_count,total_fatalities,average_fatalities
0,Battles,18880,53009,2.807680
1,Explosions/Remote violence,19313,17963,0.930099
2,Protests,25573,24,0.000938
3,Riots,1056,277,0.262311
4,Strategic developments,16059,168,0.010461
5,Violence against civilians,10560,10249,0.970549


### Reflection - Stage 4
* **What I learned:** I learned how to use multi-index aggregations to calculate multiple statistics for the same group in one pass.
* **Challenge faced:** The resulting multi-index columns were difficult to work with for downstream tasks.
* **How I solved it:** I flattened the column headers immediately after aggregation for better readability.

## Stage 5: Output

Exporting the results for analysis or machine learning.

In [6]:
output_file = "data/processed_research_data.csv"
summary_stats.to_csv(output_file, index=False)
print(f"Final results saved to {output_file}")

Final results saved to data/processed_research_data.csv


### Reflection - Stage 5
* **What I learned:** I learned the importance of disabling index saving in `to_csv` to avoid adding redundant columns to the final file.
* **Challenge faced:** Ensuring the output directory exists before saving.
* **How I solved it:** I verified the directory structure manually, but in a production script, I would use `os.makedirs(exist_ok=True)`.